In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib
import zipfile

##Uploading Dataset

In [2]:
normal = pd.read_parquet('normal.parquet')
dos = pd.read_parquet('dos.parquet')
fuzzy = pd.read_parquet('fuzzy.parquet')
gear = pd.read_parquet('gear.parquet')
rpm = pd.read_parquet('rpm.parquet')

files = {
    'normal': normal,
    'dos': dos,
    'fuzzy': fuzzy,
    'gear': gear,
    'rpm': rpm
}

for name, df in files.items():
  print(f"\n{name}: {df.shape}")
  print(df.columns.tolist())
  print(df.head())



normal: (988871, 12)
['Timestamp', 'CAN_ID', 'DLC', 'Byte0', 'Byte1', 'Byte2', 'Byte3', 'Byte4', 'Byte5', 'Byte6', 'Byte7', 'Label']
      Timestamp CAN_ID  DLC  Byte0  Byte1  Byte2  Byte3  Byte4  Byte5  Byte6  \
0  1.479121e+09   0350  8.0      5     40  132.0  102.0  109.0    0.0    0.0   
1  1.479121e+09   02c0  8.0     20      0    0.0    0.0    0.0    0.0    0.0   
2  1.479121e+09   0430  8.0      0      0    0.0    0.0    0.0    0.0    0.0   
3  1.479121e+09   04b1  8.0      0      0    0.0    0.0    0.0    0.0    0.0   
4  1.479121e+09   01f1  8.0      0      0    0.0    0.0    0.0    0.0    0.0   

   Byte7 Label  
0  162.0     R  
1    0.0     R  
2    0.0     R  
3    0.0     R  
4    0.0     R  

dos: (3665771, 12)
['Timestamp', 'CAN_ID', 'DLC', 'Byte0', 'Byte1', 'Byte2', 'Byte3', 'Byte4', 'Byte5', 'Byte6', 'Byte7', 'Label']
      Timestamp CAN_ID  DLC  Byte0  Byte1  Byte2  Byte3  Byte4  Byte5  Byte6  \
0  1.478198e+09   0316    8      5     33  104.0    9.0   33.0   33.0  

##Handling missing values

In [3]:
for name, df in files.items():
  print(f"\n{name} missing values:")
  print(df.isnull().sum())


normal missing values:
Timestamp        0
CAN_ID           0
DLC              0
Byte0            0
Byte1            0
Byte2        10129
Byte3        10129
Byte4        10129
Byte5        60735
Byte6        60735
Byte7        60735
Label            0
dtype: int64

dos missing values:
Timestamp        0
CAN_ID           0
DLC              0
Byte0            0
Byte1            0
Byte2        31188
Byte3        31188
Byte4        31188
Byte5        31188
Byte6        31188
Byte7        31188
Label            0
dtype: int64

fuzzy missing values:
Timestamp        0
CAN_ID           0
DLC              0
Byte0            0
Byte1            0
Byte2        34382
Byte3        34382
Byte4        34382
Byte5        87833
Byte6        87836
Byte7        87836
Label            0
dtype: int64

gear missing values:
Timestamp        0
CAN_ID           0
DLC              0
Byte0            0
Byte1            0
Byte2        40165
Byte3        40165
Byte4        40165
Byte5        40165
Byte6        401

In [4]:
byte_cols = [
    'Byte0', 'Byte1', 'Byte2', 'Byte3',
    'Byte4', 'Byte5', 'Byte6', 'Byte7'
]

for name, df in files.items():
  df[byte_cols] = df[byte_cols].fillna(0)
  print(name, df[byte_cols].isnull().sum().sum())


normal 0
dos 0
fuzzy 0
gear 0
rpm 0


Coverting CAN_ID from Hexadecimal to Decimal.

In [5]:
for name, df in files.items():
  print(f"\n{name}:")
  print(f"Data Type: {df['CAN_ID'].dtype}")
  print(f"Unique values: {df["CAN_ID"].nunique()}")




normal:
Data Type: object
Unique values: 27

dos:
Data Type: object
Unique values: 27

fuzzy:
Data Type: object
Unique values: 2048

gear:
Data Type: object
Unique values: 26

rpm:
Data Type: object
Unique values: 26


In [6]:
#Converting CAN_ID from Hexadecimal to Decimal numbers
for df in files.values():
  df["CAN_ID"] = df["CAN_ID"].apply(lambda x : int(str(x), 16))

normal["CAN_ID"].head()

,CAN_ID
0,848
1,704
2,1072
3,1201
4,497


Features preparation

In [16]:
feat_cols = ["CAN_ID", "DLC"] + byte_cols #10 features not including time
X_files = {}

#define features for every file
for name, df in files.items():
  X_files[name] = df[feat_cols].copy()
  print(X_files[name].shape)



(988871, 10)
(3665771, 10)
(3838860, 10)
(4443142, 10)
(4621702, 10)


In [9]:
#Data splitting 80% training 20% validation (Training data)
x_train, x_val = train_test_split(X_files["normal"], test_size=0.2, shuffle=False)

print("Training:", x_train.shape)
print("Validation:", x_val.shape)

Training: (791096, 10)
Validation: (197775, 10)


In [10]:
#scaling numeric features
scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

x_scaled_files = {}

for name in ['dos', 'fuzzy', 'gear', 'rpm']:
    x_scaled_files[name] = scaler.transform(X_files[name])

print("Train:", x_train_scaled.shape)
print("Validation:", x_val_scaled.shape)

for name, X in x_scaled_files.items():
    print(name, X.shape)

Train: (791096, 10)
Validation: (197775, 10)
dos (3665771, 10)
fuzzy (3838860, 10)
gear (4443142, 10)
rpm (4621702, 10)


In [ ]:
# Save all datasets in one ZIP file
with zipfile.ZipFile('CAN_Bus_Preprocessed_Data.zip', 'w') as zipf:

    # Save Training
    train_df = pd.DataFrame(x_train_scaled, columns=feat_cols)
    train_df.to_parquet('train_scaled.parquet', index=False)
    zipf.write('train_scaled.parquet')

    # Save Validation
    val_df = pd.DataFrame(x_val_scaled, columns=feat_cols)
    val_df.to_parquet('validation_scaled.parquet', index=False)
    zipf.write('validation_scaled.parquet')

    # Save Testing datasets with Labels
    for name, X in x_scaled_files.items():
        test_df = pd.DataFrame(X, columns=feat_cols)
        test_df['Label'] = files[name]['Label'].values

        filename = f'{name}_test_scaled.parquet'
        test_df.to_parquet(filename, index=False)
        zipf.write(filename)


# Download ZIP
from google.colab import files
files.download('CAN_Bus_Preprocessed_Data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trying Time features

In [14]:
#Adding Time features
for name, df in files.items():
  df.sort_values('Timestamp', inplace=True)

  #Time differnce between every message
  df['time_diff'] = df['Timestamp'].diff().fillna(0)

  #time difference between every message with the same CAN_ID
  df["can_id_time_diff"] = (
      df.groupby("CAN_ID")["Timestamp"].diff().fillna(0)
  )

In [18]:
#Adding new time features
feat_cols1 = ["CAN_ID", "DLC"] + byte_cols +["time_diff", "can_id_time_diff"] #12 features including time
X_files1 = {}

for name, df in files.items():
  X_files1[name] = df[feat_cols1].copy()
  print(X_files1[name].shape)

(988871, 12)
(3665771, 12)
(3838860, 12)
(4443142, 12)
(4621702, 12)


In [19]:
#Data splitting 80% training 20% validation (Training data)
x_train1, x_val1 = train_test_split(X_files1["normal"], test_size=0.2, shuffle=False)

print("Training:", x_train1.shape)
print("Validation:", x_val1.shape)

Training: (791096, 12)
Validation: (197775, 12)


In [22]:
scaler1 = MinMaxScaler()

x_train_scaled1 = scaler1.fit_transform(x_train1)
x_val_scaled1 = scaler1.transform(x_val1)

x_scaled_files1 = {}

for name in ['dos', 'fuzzy', 'gear', 'rpm']:
    x_scaled_files1[name] = scaler1.transform(X_files1[name])

print("Train:", x_train_scaled1.shape)
print("Validation:", x_val_scaled1.shape)

for name, X in x_scaled_files1.items():
    print(name, X.shape)

Train: (791096, 12)
Validation: (197775, 12)
dos (3665771, 12)
fuzzy (3838860, 12)
gear (4443142, 12)
rpm (4621702, 12)


In [25]:
# Save all datasets in one ZIP file
with zipfile.ZipFile('CAN_Bus_Preprocessed_Data_Time_Features.zip', 'w') as zipf:

    # Save Training
    train_df = pd.DataFrame(x_train_scaled1, columns=feat_cols1)
    train_df.to_parquet('train_scaled_time.parquet', index=False)
    zipf.write('train_scaled_time.parquet')

    # Save Validation
    val_df = pd.DataFrame(x_val_scaled1, columns=feat_cols1)
    val_df.to_parquet('validation_scaled_time.parquet', index=False)
    zipf.write('validation_scaled_time.parquet')

    # Save Testing datasets with Labels
    for name, X in x_scaled_files1.items():
        test_df = pd.DataFrame(X, columns=feat_cols1)
        test_df['Label'] = files[name]['Label'].values

        filename = f'{name}_test_scaled_time.parquet'
        test_df.to_parquet(filename, index=False)
        zipf.write(filename)

    # Save Scaler
    joblib.dump(scaler1, 'scaler_time.pkl')
    zipf.write('scaler_time.pkl')

# Download ZIP
from google.colab import files
files.download('CAN_Bus_Preprocessed_Data_Time_Features.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>